# Kaggle — train and evaluate the YOLO models

**Settings:** Accelerator **GPU T4 x2**, Internet **on**, Persistence **Files only**
(so `runs/` survives a restart and training resumes).
**Input:** attach the Kaggle Dataset you uploaded from `data/vehicles/` (any name works; the
cell below finds it).

Each model: train 100 epochs on both T4s → copy best weights → evaluate on the test split
(mAP + speed on a single T4). Outputs land in `/kaggle/working/` → download `outputs.zip`.
One `s` model takes roughly an hour; run a few per session.

In [ ]:
REPO = "https://github.com/Ananth170404/traffic-detection-benchmark.git"
%cd /kaggle/working
!test -d traffic-detection-benchmark || git clone -q $REPO
%cd /kaggle/working/traffic-detection-benchmark
!git pull -q
!pip install -q -e ".[train]"

In [ ]:
from pathlib import Path

raw = next(Path("/kaggle/input").rglob("data.yaml")).parent  # the uploaded data/vehicles
print("raw dataset:", raw)
!python scripts/prepare_data.py --raw "{raw}" --out data/processed

## Choose what to run this session

Model keys come from `configs/train/yolo.yaml`. Set `SMOKE = True` first to check everything
works with 1 epoch, then set it back to `False`.

In [ ]:
VARIANT = "v3"                       # "v3" (car/bus/truck) or "v12" (original classes)
MODELS = ["yolov8s", "yolo11s", "yolo26s"]
TRAIN_YOLOV5 = True
SMOKE = False

epochs = "--epochs 1" if SMOKE else ""
data = f"data/processed/{VARIANT}"
EVAL = "--opt conf=0.001 --opt iou=0.6 --opt max_det=300 --opt device=0"

In [ ]:
for m in MODELS:
    name = f"{m}_{VARIANT}"
    !python scripts/train_yolo.py --model {m} --data {data} --device 0,1 {epochs}
    !python scripts/evaluate.py --detector tdb.models.yolo:UltralyticsDetector \
        --weights weights/{name}.pt --data {data} --split test --name {name} {EVAL}

In [ ]:
if TRAIN_YOLOV5:
    name = f"yolov5s_{VARIANT}"
    !python scripts/train_yolov5.py --data {data} --gpus 2 {epochs}
    !python scripts/evaluate.py --detector tdb.models.yolo:YOLOv5Detector \
        --weights weights/{name}.pt --data {data} --split test --name {name} \
        {EVAL} --opt repo=yolov5

## Summary and download

In [ ]:
import json

import pandas as pd

rows = []
for f in sorted(Path("results").glob("*.json")):
    r = json.loads(f.read_text())
    rows.append({"model": r["name"], "mAP50": r["metrics"]["map50"],
                 "mAP50-95": r["metrics"]["map50_95"],
                 "ms/img": (r["speed"] or {}).get("latency_ms_mean"),
                 "FPS": (r["speed"] or {}).get("fps"),
                 "params (M)": (r["model"].get("params") or 0) / 1e6})
pd.DataFrame(rows).round(3)

In [ ]:
# results/ (small, commit to git) + weights/ (GitHub Release) + training curves
!zip -qr /kaggle/working/outputs.zip results weights runs -x "runs/*/*/weights/last.pt"
!ls -lh /kaggle/working/outputs.zip